# Captura de imágenes
Libreta usada exclusivamente para capturar imagenes bloqueando el rostro de las personas en video

In [6]:
import cv2

entrada = "video_original.mp4"
salida = "video_anonimizado.mp4"

# Detector frontal incluido con OpenCV.
# No detectará todos los rostros en todas las condiciones.
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(1)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {entrada}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    salida,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (ancho, alto)
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("No se pudo crear el video de salida.")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        rostros = face_cascade.detectMultiScale(
            gris,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        # Margen alrededor del rostro para cubrir algo más que la cara.
        for (x, y, w, h) in rostros:
            margen_x = int(0.15 * w)
            margen_y = int(0.20 * h)
            x1 = max(0, x - margen_x)
            y1 = max(0, y - margen_y)
            x2 = min(ancho, x + w + margen_x)
            y2 = min(alto, y + h + margen_y)

            # Máscara opaca
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), thickness=-1)

        writer.write(frame)

        # Vista en tiempo real del video ya procesado
        cv2.imshow("Video anonimizado - presiona q para salir", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

print("Procesamiento terminado.")


Procesamiento terminado.


## Usando script

In [ ]:
from pathlib import Path
import sys

# Busca el script desde la raíz del repositorio o desde DataCapture.
_candidates = (Path.cwd() / "DataCapture", Path.cwd())
_module_dir = next(
    (path.resolve() for path in _candidates if (path / "capture_wo_pii.py").is_file()),
    None,
)
if _module_dir is None:
    raise FileNotFoundError("No se encontró DataCapture/capture_wo_pii.py desde el directorio actual.")
sys.path.insert(0, str(_module_dir))

from capture_wo_pii import capture_video

# Cambia source=1 para otra cámara. 's' guarda una captura enmascarada; 'q' termina.
capture_video(source=1, padding=0.30)